# 1. Base Setup

## 1.1 Install packages

In [10]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
!pip install kagglehub
!pip install statsmodels
!pip install xgboost
!pip install catboost

## 1.2 Load all needed imports

In [12]:
from pathlib import Path

import pandas as pd
import kagglehub
from kagglehub import KaggleDatasetAdapter
import numpy as np
import matplotlib.pyplot as plt

from dateutil.relativedelta import relativedelta

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, log_loss, confusion_matrix, accuracy_score, f1_score
from sklearn.calibration import calibration_curve
from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit, GridSearchCV

from scipy.stats import randint, uniform
import sys, os
sys.path.append(os.path.abspath(".."))

# 2. Data Cleaning and preprocessing


In [13]:
import cf_copilot
print(cf_copilot.__file__)

/home/jeroenewalts/code/EwaltsJ/cf_copilot/cf_copilot/__init__.py


In [14]:
def load_cashflow_data(csv_name: str = "dataset.csv") -> pd.DataFrame:
    """Load invoice dataset from local raw_data folder, or download from Kaggle.

    Args:
        csv_name: filename of the CSV to load.

    Returns:
        A pandas DataFrame with the raw invoice data.
    """
    base_dir = Path.cwd().parent
    raw_data_dir = base_dir / "raw_data"
    raw_data_dir.mkdir(parents=True, exist_ok=True)
    local_path = raw_data_dir / csv_name

    if local_path.is_file():
        print(f"Loading local file: {local_path}")
        return pd.read_csv(local_path)

    print("Local file not found, downloading from Kaggle via kagglehub...")
    df = kagglehub.dataset_load(
        KaggleDatasetAdapter.PANDAS,
        "pradumn203/payment-date-prediction-for-invoices-dataset",
        "dataset.csv",
    )

    df.to_csv(local_path, index=False)
    print(f"Saved local copy to {local_path}")

    return df

In [15]:
from cf_copilot.ml_logic.data import data_cleaning, build_sliding_window_snapshots
from cf_copilot.ml_logic.encoders import preprocess, NUMERIC_FEATURES, CATEGORICAL_FEATURES
from cf_copilot.ml_logic.model import initialize_model, train_model

hist_df = load_cashflow_data()

df = data_cleaning(hist_df)
big_df = build_sliding_window_snapshots(df)

big_df = big_df.sort_values("invoice_sent").reset_index(drop=True)

# Train+valid for search, final holdout for test
valid_cutoff = big_df["reference_date"].quantile(0.6)
test_cutoff = big_df["reference_date"].quantile(0.8)

search_df = big_df[big_df["reference_date"] <= test_cutoff].copy()
final_test_df = big_df[big_df["reference_date"] > test_cutoff].copy()

# Explicit train / validation split inside search_df
train_df = search_df[search_df["reference_date"] <= valid_cutoff].copy()
valid_df = search_df[search_df["reference_date"] > valid_cutoff].copy()

# Preprocess each split
X_search_xgb, y_search_xgb = preprocess(search_df)
X_train_xgb, y_train_xgb = preprocess(train_df)
X_valid_xgb, y_valid_xgb = preprocess(valid_df)
X_final_test_xgb, y_final_test_xgb = preprocess(final_test_df)

# XGBoost needs zero-based labels
y_search_xgb = y_search_xgb - 1
y_train_xgb = y_train_xgb - 1
y_valid_xgb = y_valid_xgb - 1
y_final_test_xgb = y_final_test_xgb - 1

print("Search:", X_search_xgb.shape, y_search_xgb.shape)
print("Train :", X_train_xgb.shape, y_train_xgb.shape)
print("Valid :", X_valid_xgb.shape, y_valid_xgb.shape)
print("Test  :", X_final_test_xgb.shape, y_final_test_xgb.shape)

Loading local file: /home/jeroenewalts/code/EwaltsJ/cf_copilot/raw_data/dataset.csv
Original rows: 39152
Augmented rows: 98169
week_bucket
1    38825
2    33060
3    10401
4     4009
5     3172
6     2192
7     6510
Name: count, dtype: int64
Search: (79725, 18) (79725,)
Train : (60018, 18) (60018,)
Valid : (19707, 18) (19707,)
Test  : (18444, 18) (18444,)


In [16]:
numeric_transformer = SimpleImputer(strategy="median")

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="constant", fill_value=-1)),
    ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, NUMERIC_FEATURES),
        ("cat", categorical_transformer, CATEGORICAL_FEATURES),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['business_year', 'invoice_age_days',
                                  'days_until_due', 'pay_terms_days',
                                  'invoice_month', 'due_month', 'days_past_due',
                                  'customer_avg_delay', 'late_payment_ratio',
                                  'prev_transaction_count',
                                  'days_since_last_invoice',
                                  'customer_risk_score', 'invoice_amount',
                                  'invoice_amount_log']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(fill_value=-1,
                                                                strategy='constant')),
                                                 ('encoder',
                                                  OrdinalEncoder(handle_unknown='use_encoded_value',
                                                                 unknown_value=-1))]),
                                 ['invoice_currency', 'document_type',
                                  'invoice_size_cat', 'invoice_size_cat_q'])])

In [17]:
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
import datetime

xgb_refine_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(
        objective="multi:softprob",
        num_class=7,
        eval_metric="mlogloss",
        random_state=42,
        n_jobs=4,
        tree_method="hist",
        verbosity=0,
        colsample_bytree=0.7265477506155757,
        gamma=0.058794858725743554,
        max_depth=7,
        min_child_weight=2,
        reg_alpha=0.5867511656638482,
        reg_lambda=1.930510614528276,
        subsample=0.8821102743060054,
    ))
])

In [18]:
param_grid = {
    "classifier__learning_rate": [0.018, 0.022, 0.0245, 0.028, 0.032],
    "classifier__n_estimators": [280, 320, 352, 400, 450],
}

print("START REFINEMENT:", datetime.datetime.now())

grid_search = GridSearchCV(
    estimator=xgb_refine_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_log_loss",
    n_jobs=4,
    verbose=2,
    return_train_score=True,
)

grid_search.fit(X_search_xgb, y_search_xgb)

best_model = grid_search.best_estimator_

print("END REFINEMENT:", datetime.datetime.now())
print("Best CV log_loss:", -grid_search.best_score_)
print("Best params:", grid_search.best_params_)

START REFINEMENT: 2026-03-25 13:49:38.743780
Fitting 5 folds for each of 25 candidates, totalling 125 fits
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=280; total time=   6.3s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=280; total time=   6.4s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=280; total time=   6.5s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=280; total time=   6.5s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=280; total time=   5.8s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=320; total time=   6.6s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=320; total time=   6.6s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=320; total time=   6.8s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=320; total time=   6.4s
[CV] END classifier__learning_rate=0.018, classifier__n_estimators=320; to

In [19]:
y_proba_refined = best_model.predict_proba(X_final_test_xgb)
y_pred_refined = best_model.predict(X_final_test_xgb)

clf_classes_refined = np.array(best_model.named_steps["classifier"].classes_)

refined_log_loss = log_loss(y_final_test_xgb, y_proba_refined, labels=clf_classes_refined)
refined_accuracy = accuracy_score(y_final_test_xgb, y_pred_refined)
refined_f1_macro = f1_score(y_final_test_xgb, y_pred_refined, average="macro")

print("\nRefined XGBoost holdout performance")
print(f"log_loss  : {refined_log_loss:.6f}")
print(f"accuracy  : {refined_accuracy:.6f}")
print(f"f1_macro  : {refined_f1_macro:.6f}")


Refined XGBoost holdout performance
log_loss  : 0.698716
accuracy  : 0.754771
f1_macro  : 0.557127


In [20]:
from cf_copilot.cashflow_prediction.evaluation import (
    build_actual_weekly_cf,
    compare_forecast_vs_actual,
    compute_forecast_metrics,
)
from cf_copilot.cashflow_prediction.registry import WEEK_CLASSES

def apply_probability_tilt(probas: np.ndarray, class_weights: np.ndarray) -> np.ndarray:
    """
    Multiply class probabilities by deterministic class weights,
    then renormalize row-wise so probabilities sum to 1.
    """
    weighted = probas * class_weights
    row_sums = weighted.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    return weighted / row_sums


def build_weekly_forecast_from_xgb_proba(
    probas: np.ndarray,
    snapshot_df: pd.DataFrame,
    class_weights: np.ndarray | None = None,
) -> pd.DataFrame:
    """
    Build weekly forecast from XGBoost probabilities.
    XGBoost classes are 0..6, where:
      0->week1, 1->week2, ..., 5->week6, 6->class7(outside horizon)
    """
    if class_weights is not None:
        probas = apply_probability_tilt(probas, class_weights)

    pred_cash_df = snapshot_df.copy()

    for week in WEEK_CLASSES:  # 1..6
        class_idx = week - 1
        pred_cash_df[f"p_{week}"] = probas[:, class_idx]

    weekly_forecast_df = pd.DataFrame([
        {
            "week_bucket": int(w),
            "forecast_cash": round(
                float((pred_cash_df["total_open_amount"] * pred_cash_df[f"p_{w}"]).sum()),
                2,
            ),
        }
        for w in WEEK_CLASSES
    ])

    return weekly_forecast_df


def evaluate_forecast_holdout_with_bias(
    model,
    test_df: pd.DataFrame,
    class_weights: np.ndarray | None = None,
    verbose: bool = True,
):
    """
    Forecast evaluation for XGBoost with optional deterministic probability tilt.
    """
    per_reference_results = []

    for reference_date, snapshot_df in test_df.groupby("reference_date"):
        snapshot_df = snapshot_df.copy()
        if len(snapshot_df) == 0:
            continue

        X_snapshot, _ = preprocess(snapshot_df)
        probas = model.predict_proba(X_snapshot)

        weekly_forecast_df = build_weekly_forecast_from_xgb_proba(
            probas=probas,
            snapshot_df=snapshot_df,
            class_weights=class_weights,
        )

        actual_weekly_df = build_actual_weekly_cf(
            invoices_df=snapshot_df,
            reference_date=pd.Timestamp(reference_date),
        )

        comparison_df = compare_forecast_vs_actual(
            weekly_forecast_df=weekly_forecast_df,
            actual_weekly_df=actual_weekly_df,
        )

        snapshot_metrics = compute_forecast_metrics(comparison_df)

        per_reference_results.append(snapshot_metrics)

    mae_values = [r["MAE (weekly)"] for r in per_reference_results]
    mape_values = [r["MAPE (weekly)"] for r in per_reference_results if pd.notna(r["MAPE (weekly)"])]
    total_actual_values = [r["Total actual cf"] for r in per_reference_results]
    total_forecast_values = [r["Total forecast cf"] for r in per_reference_results]
    total_diff_values = [r["Total cf difference"] for r in per_reference_results]

    forecast_metrics = {
        "forecast_mae_weekly": float(np.mean(mae_values)),
        "forecast_mape_weekly": float(np.mean(mape_values)) if mape_values else np.nan,
        "forecast_total_actual_cf": float(np.mean(total_actual_values)),
        "forecast_total_forecast_cf": float(np.mean(total_forecast_values)),
        "forecast_total_cf_difference": float(np.mean(total_diff_values)),
    }

    if verbose:
        print(f"✅ Forecast MAE weekly: {forecast_metrics['forecast_mae_weekly']:.2f}")
        print(f"✅ Forecast MAPE weekly: {forecast_metrics['forecast_mape_weekly']:.4f}")
        print(f"✅ Forecast total actual cf: {forecast_metrics['forecast_total_actual_cf']:.2f}")
        print(f"✅ Forecast total forecast cf: {forecast_metrics['forecast_total_forecast_cf']:.2f}")
        print(f"✅ Forecast total cf difference: {forecast_metrics['forecast_total_cf_difference']:.2f}")

    return forecast_metrics

In [21]:
# Fit train-only XGBoost using the refined best parameters
best_clf_params = best_model.named_steps["classifier"].get_params()

xgb_train_only = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(**best_clf_params))
])

print("START TRAIN-ONLY FIT FOR BIAS TUNING:", datetime.datetime.now())
xgb_train_only.fit(X_train_xgb, y_train_xgb)
print("END TRAIN-ONLY FIT FOR BIAS TUNING:", datetime.datetime.now())

START TRAIN-ONLY FIT FOR BIAS TUNING: 2026-03-25 13:54:11.173532
END TRAIN-ONLY FIT FOR BIAS TUNING: 2026-03-25 13:54:14.800953


In [22]:
# Class order for XGBoost:
# [week1, week2, week3, week4, week5, week6, class7/outside horizon]

bias_grid = {
    "neutral": np.array([1.00, 1.00, 1.00, 1.00, 1.00, 1.00, 1.00]),
    "early_small": np.array([1.05, 1.03, 1.02, 1.00, 0.98, 0.96, 1.00]),
    "early_medium": np.array([1.10, 1.06, 1.03, 1.00, 0.96, 0.92, 1.00]),
    "late_small": np.array([0.96, 0.98, 1.00, 1.02, 1.04, 1.06, 1.00]),
    "deemphasize_outside": np.array([1.02, 1.02, 1.01, 1.00, 0.99, 0.98, 0.95]),
    "slight_early_and_less_outside": np.array([1.05, 1.03, 1.01, 1.00, 0.98, 0.97, 0.95]),
}

bias_results = []

for bias_name, class_weights in bias_grid.items():
    print(f"\nTesting bias: {bias_name}")

    metrics = evaluate_forecast_holdout_with_bias(
        model=xgb_train_only,
        test_df=valid_df,
        class_weights=class_weights,
        verbose=False,
    )

    bias_results.append({
        "bias_name": bias_name,
        "class_weights": class_weights,
        **metrics,
    })

bias_results_df = pd.DataFrame(bias_results).sort_values(
    by=["forecast_mae_weekly", "forecast_mape_weekly"]
).reset_index(drop=True)

bias_results_df


Testing bias: neutral

Testing bias: early_small

Testing bias: early_medium

Testing bias: late_small

Testing bias: deemphasize_outside

Testing bias: slight_early_and_less_outside


,bias_name,class_weights,forecast_mae_weekly,forecast_mape_weekly,forecast_total_actual_cf,forecast_total_forecast_cf,forecast_total_cf_difference
0,slight_early_and_less_outside,"[1.05, 1.03, 1.01, 1.0, 0.98, 0.97, 0.95]",760336.462727,0.205455,5.191058e+07,5.136382e+07,-546755.540909
1,neutral,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]",761153.067273,0.209091,5.191058e+07,5.128589e+07,-624693.352727
2,early_small,"[1.05, 1.03, 1.02, 1.0, 0.98, 0.96, 1.0]",761524.517273,0.205455,5.191058e+07,5.130493e+07,-605652.895455
3,early_medium,"[1.1, 1.06, 1.03, 1.0, 0.96, 0.92, 1.0]",763019.270909,0.200000,5.191058e+07,5.132094e+07,-589637.718182
4,deemphasize_outside,"[1.02, 1.02, 1.01, 1.0, 0.99, 0.98, 0.95]",763336.621818,0.208182,5.191058e+07,5.135506e+07,-555520.011818
5,late_small,"[0.96, 0.98, 1.0, 1.02, 1.04, 1.06, 1.0]",768748.221818,0.217273,5.191058e+07,5.128330e+07,-627276.540909


In [23]:
best_bias_name = bias_results_df.iloc[0]["bias_name"]
best_class_weights = bias_results_df.iloc[0]["class_weights"]

print("Best bias:", best_bias_name)
print("Best class weights:", best_class_weights)

forecast_metrics_refined_raw = evaluate_forecast_holdout_with_bias(
    model=best_model,
    test_df=final_test_df,
    class_weights=None,
    verbose=True,
)

forecast_metrics_refined_biased = evaluate_forecast_holdout_with_bias(
    model=best_model,
    test_df=final_test_df,
    class_weights=best_class_weights,
    verbose=True,
)

Best bias: slight_early_and_less_outside
Best class weights: [1.05 1.03 1.01 1.   0.98 0.97 0.95]
✅ Forecast MAE weekly: 536671.33
✅ Forecast MAPE weekly: 0.3285
✅ Forecast total actual cf: 27906118.13
✅ Forecast total forecast cf: 27744114.75
✅ Forecast total cf difference: -162003.38
✅ Forecast MAE weekly: 533328.57
✅ Forecast MAPE weekly: 0.3235
✅ Forecast total actual cf: 27906118.13
✅ Forecast total forecast cf: 27775178.23
✅ Forecast total cf difference: -130939.90


In [24]:
comparison_df = pd.DataFrame([
    {
        "model": "XGBoost_previous_best",
        "log_loss": 0.703987,
        "accuracy": 0.756940,
        "f1_macro": 0.568073,
        "forecast_mae_weekly": 535341.07,
        "forecast_mape_weekly": 0.325,
        "forecast_total_cf_difference": -81844.183,
    },
    {
        "model": "XGBoost_refined_raw",
        "log_loss": refined_log_loss,
        "accuracy": refined_accuracy,
        "f1_macro": refined_f1_macro,
        "forecast_mae_weekly": forecast_metrics_refined_raw["forecast_mae_weekly"],
        "forecast_mape_weekly": forecast_metrics_refined_raw["forecast_mape_weekly"],
        "forecast_total_cf_difference": forecast_metrics_refined_raw["forecast_total_cf_difference"],
    },
    {
        "model": f"XGBoost_refined_biased_{best_bias_name}",
        "log_loss": refined_log_loss,   # same classifier, business bias only affects forecast layer
        "accuracy": refined_accuracy,   # same classifier
        "f1_macro": refined_f1_macro,   # same classifier
        "forecast_mae_weekly": forecast_metrics_refined_biased["forecast_mae_weekly"],
        "forecast_mape_weekly": forecast_metrics_refined_biased["forecast_mape_weekly"],
        "forecast_total_cf_difference": forecast_metrics_refined_biased["forecast_total_cf_difference"],
    },
])

comparison_df

,model,log_loss,accuracy,f1_macro,forecast_mae_weekly,forecast_mape_weekly,forecast_total_cf_difference
0,XGBoost_previous_best,0.703987,0.756940,0.568073,535341.0700,0.3250,-81844.183
1,XGBoost_refined_raw,0.698716,0.754771,0.557127,536671.3255,0.3285,-162003.380
2,XGBoost_refined_biased_slight_early_and_less_o...,0.698716,0.754771,0.557127,533328.5660,0.3235,-130939.899
